In [1]:
import pandas as pd
import os
import re

DATA_DIR = '../数据'
df = pd.read_csv(os.path.join(DATA_DIR, 'raw_jobs.csv'))
df.shape

(3200, 11)

In [2]:
df.isnull().sum()

岗位名称     0
薪资范围    28
工作城市    10
学历要求    21
工作经验     0
发布时间     0
公司名称     0
行业类别     0
公司性质     0
公司规模    39
技能标签    62
dtype: int64

#### 去重

In [3]:
df = df.drop_duplicates(subset=['岗位名称', '公司名称', '薪资范围']).reset_index(drop=True)
df.shape

(2263, 11)

#### 处理缺失值

In [4]:
df = df.dropna(subset=['薪资范围']).reset_index(drop=True)
df['工作城市'] = df['工作城市'].fillna('未知')
df['学历要求'] = df['学历要求'].fillna('不限')
df['公司规模'] = df['公司规模'].fillna('未知')
df['技能标签'] = df['技能标签'].fillna('')
df.isnull().sum()

岗位名称    0
薪资范围    0
工作城市    0
学历要求    0
工作经验    0
发布时间    0
公司名称    0
行业类别    0
公司性质    0
公司规模    0
技能标签    0
dtype: int64

In [5]:
df.shape

(2243, 11)

#### 薪资解析

In [6]:
def parse_salary(text):
    is_annual = '/年' in text
    bonus_match = re.search(r'(\d+)薪', text)
    bonus = int(bonus_match.group(1)) if bonus_match else 12
    range_match = re.search(r'(\d+\.?\d*)(千|万)?-(\d+\.?\d*)(千|万)', text)
    if not range_match:
        return pd.Series([None, None, None, bonus])
    low_num, low_unit, high_num, high_unit = range_match.groups()
    low_unit = low_unit or high_unit
    low = float(low_num) * (1 if low_unit == '万' else 0.1)
    high = float(high_num) * (1 if high_unit == '万' else 0.1)
    if is_annual:
        low, high = low / 12, high / 12
    return pd.Series([low, high, (low + high) / 2, bonus])

df[['月薪下限', '月薪上限', '月薪均值', '薪资月数']] = df['薪资范围'].apply(parse_salary)
df['月薪均值'].isnull().sum()

np.int64(11)

In [7]:
df = df.dropna(subset=['月薪均值']).reset_index(drop=True)
df.shape

(2232, 15)

#### 过滤薪资异常值

In [8]:
q1, q3 = df['月薪均值'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 3 * iqr
lower = max(q1 - 3 * iqr, 0)
df = df[(df['月薪均值'] >= lower) & (df['月薪均值'] <= upper)].reset_index(drop=True)
df.shape

(2210, 15)

In [9]:
df['月薪均值'].describe()

count    2210.000000
mean        1.828903
std         0.881482
min         0.300000
25%         1.200000
50%         1.650000
75%         2.250000
max         5.250000
Name: 月薪均值, dtype: float64

#### 工作经验解析

In [10]:
def parse_experience(text):
    if text == '无需经验':
        return 0
    nums = re.findall(r'\d+\.?\d*', text)
    if len(nums) == 1:
        return float(nums[0])
    return (float(nums[0]) + float(nums[1])) / 2

df['经验年限'] = df['工作经验'].apply(parse_experience)
df['经验年限'].describe()

count    2210.000000
mean        2.702489
std         1.786808
min         0.000000
25%         2.000000
50%         3.000000
75%         3.500000
max        10.000000
Name: 经验年限, dtype: float64

In [11]:
df['城市'] = df['工作城市'].str.split('·').str[0]
df['城市'].value_counts()

城市
深圳    316
上海    305
广州    292
北京    286
武汉    274
杭州    251
成都    247
南京    225
未知      8
重庆      3
南宁      3
Name: count, dtype: int64

In [12]:
target_cities = ['北京', '上海', '广州', '深圳', '杭州', '成都', '武汉', '南京']
df = df[df['城市'].isin(target_cities)].reset_index(drop=True)
df.shape

(2196, 17)

#### 非数值类型特征编码

In [13]:
edu_order = {'不限': 0, '中技/中专': 1, '大专': 2, '本科': 3, '硕士': 4, '博士': 5}
df['学历编码'] = df['学历要求'].map(edu_order)

size_order = {'未知': 0, '少于50人': 1, '50-150人': 2, '150-500人': 3, '500-1000人': 4, '1000-5000人': 5, '5000-10000人': 6, '10000人以上': 7}
df['规模编码'] = df['公司规模'].map(size_order)

In [14]:
top_industries = df['行业类别'].value_counts().head(15).index
df['行业分组'] = df['行业类别'].where(df['行业类别'].isin(top_industries), '其他')
df['行业分组'].value_counts()

行业分组
其他             1103
计算机软件           229
制药/生物工程         129
电子/半导体/集成电路     112
人工智能             95
医疗设备/器械          65
人力资源服务           64
机械/设备/重工         62
汽车研发/制造          56
仪器仪表/工业自动化       51
互联网/电子商务         49
计算机服务            45
新能源              42
保险               35
其他行业             31
通信/网络设备          28
Name: count, dtype: int64

In [15]:
df = pd.get_dummies(df, columns=['城市', '行业分组', '公司性质'])
df.shape

(2196, 51)

#### 数值标准化

In [17]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['经验年限', '学历编码', '规模编码', '薪资月数']
scaler = StandardScaler()
scaled = scaler.fit_transform(df[scale_cols])
for i, col in enumerate(scale_cols):
    df[col + '_标准化'] = scaled[:, i]
df[[c + '_标准化' for c in scale_cols]].describe().round(4)

,经验年限_标准化,学历编码_标准化,规模编码_标准化,薪资月数_标准化
count,2196.0000,2196.0000,2196.0000,2196.0000
mean,-0.0000,0.0000,0.0000,-0.0000
std,1.0002,1.0002,1.0002,1.0002
min,-1.5196,-5.0914,-1.8453,-0.4803
25%,-0.3987,-0.1833,-0.8136,-0.4803
50%,0.1618,-0.1833,-0.2978,-0.4803
75%,0.4420,-0.1833,0.7338,0.6501
max,4.0851,3.0888,1.7654,8.5629


In [18]:
df.to_csv(os.path.join(DATA_DIR, 'processed_jobs.csv'), index=False, encoding='utf-8-sig')
df.shape

(2196, 55)